# Siamese Inference Demo (Local + Google Colab)\n\nThis notebook runs pairwise signature verification using the final Siamese checkpoint.\n\nIf opened in Google Colab, setup runs automatically from `IN_COLAB` detection.\n\nDecision rule:\n- `match` if distance <= threshold\n- `non-match` otherwise\n

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
print("Environment:", "Google Colab" if IN_COLAB else "Local Python")

REPO_URL = "https://github.com/smondal13/CSE60868-signature-project.git"
REPO_DIRNAME = "CSE60868-signature-project"

if IN_COLAB:
    repo_target = Path("/content") / REPO_DIRNAME
    if not repo_target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_target)], check=True)
    else:
        print(f"Repo already exists at {repo_target}; skipping clone.")

    # Colab already ships with torch; avoid reinstalling torch/torchvision.
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "numpy>=1.24", "pandas>=2.1", "Pillow>=9.5", "matplotlib>=3.8"
    ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(repo_target)], check=True)
    print("Colab dependencies ready.")


Environment: Local Python


In [25]:
from pathlib import Path
import sys
import torch
import pandas as pd

def find_repo_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    # Common Colab clone location
    candidates.append(Path("/content/CSE60868-signature-project"))
    for candidate in candidates:
        if (candidate / "README.md").exists() and (candidate / "siamese" / "src").exists():
            return candidate
    raise RuntimeError(
        "Could not find repo root. If on Colab, either clone the repo first "
        "or set RUN_COLAB_SETUP=True in the previous cell."
    )

REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)

SRC_ROOT = REPO_ROOT / "siamese" / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from signature_siamese.checkpoints import load_checkpoint, infer_embedding_dim
from signature_siamese.data.transforms import SignaturePreprocessor
from signature_siamese.model.siamese import SiameseNetwork

print("Repo root:", REPO_ROOT)

# Paths and runtime defaults
CHECKPOINT_PATH = REPO_ROOT / "siamese" / "runs" / "siamese_full_hpo_lr5e4_m075_20260408-152352" / "checkpoints" / "best.pt"
IMAGE_HEIGHT = 155
IMAGE_WIDTH = 220
DEVICE = torch.device("cpu")

assert CHECKPOINT_PATH.exists(), f"Missing checkpoint: {CHECKPOINT_PATH}"
print("Checkpoint:", CHECKPOINT_PATH)


Repo root: /Users/smondal/Documents/Github/CSE60868-signature-project
Checkpoint: /Users/smondal/Documents/Github/CSE60868-signature-project/siamese/runs/siamese_full_hpo_lr5e4_m075_20260408-152352/checkpoints/best.pt


In [26]:
# Load model and threshold from checkpoint
checkpoint = load_checkpoint(CHECKPOINT_PATH)
embedding_dim = infer_embedding_dim(checkpoint, fallback=128)
threshold_bhsig = float(checkpoint["best_threshold"])

model = SiameseNetwork(embedding_dim=embedding_dim)
model.load_state_dict(checkpoint["model_state"])
model.to(DEVICE)
model.eval()

preprocessor = SignaturePreprocessor(image_height=IMAGE_HEIGHT, image_width=IMAGE_WIDTH)

print("Embedding dim:", embedding_dim)
print("BHSig locked threshold:", threshold_bhsig)


Embedding dim: 128
BHSig locked threshold: 0.5195213260507225


In [27]:
def pair_distance(image_a_path: Path, image_b_path: Path) -> float:
    image_a = preprocessor(image_a_path).unsqueeze(0).to(DEVICE)
    image_b = preprocessor(image_b_path).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb_a = model.forward_once(image_a)
        emb_b = model.forward_once(image_b)
        return float(torch.norm(emb_a - emb_b, p=2, dim=1).item())

def predict(distance: float, threshold: float) -> str:
    return "match" if distance <= threshold else "non-match"


In [28]:
# Compare under two thresholds (Part 4 pair first, team pairs optional)
threshold_cedar = 0.09396831710459949  # Mode-B calibrated threshold used in report

pairs = [
    # Part 4 official validation demo pair
    ("B-S-83-G-04 vs B-S-83-F-05", REPO_ROOT / "validation" / "B-S-83-G-04.tif", REPO_ROOT / "validation" / "B-S-83-F-05.tif", "non-match"),
]

rows = []
for label, a_path, b_path, expected in pairs:
    assert a_path.exists(), f"Missing: {a_path}"
    assert b_path.exists(), f"Missing: {b_path}"
    d = pair_distance(a_path, b_path)
    pred_bhsig = predict(d, threshold_bhsig)
    pred_cedar = predict(d, threshold_cedar)
    rows.append({
        "pair": label,
        "distance": round(d, 6),
        "expected": expected,
        "pred_bhsig": pred_bhsig,
        "correct_bhsig": pred_bhsig == expected,
    })

pd.DataFrame(rows)


,pair,distance,expected,pred_bhsig,correct_bhsig
0,B-S-83-G-04 vs B-S-83-F-05,0.561213,non-match,non-match,True
